Product Mapping
===============

Purpose:
The product IDs used by the SimCompanies market website differ from
the internal product IDs stored in Book1.xlsx.

The market resource IDs and corresponding product names were manually
collected from the game's market pages. This script maps these external
IDs to the internal product table using product names.

Process:
1. Load internal products from Book1.xlsx.
2. Load the collected market resource IDs and product names.
3. Normalize product names to improve matching consistency.
4. Perform a FULL OUTER JOIN to identify matching and unmatched products.
5. Review unmatched records and correct potential spelling or naming
   inconsistencies manually.
6. Save the mapping results with a INNER JOIN and validation reports to Excel.

Note:
Product names are currently used as the matching key. The resulting
mapping should be reviewed before being used in further data processing.

In [1]:
import pandas as pd
import numpy as np

In [14]:
market_ids = '''1 - power
2 - water
3 - apples
4 - oranges
5 - grapes
6 - grain
7 - steak
8 - sausages
9 - eggs
10 - crude oil
11 - petrol
12 - diesel
13 - transport
14 - minerals
15 - bauxite
16 - silicon
17 - chemicals
18 - aluminium
19 - plastic
20 - processors
21 - electronic components
22 - batteries
23 - displays
24 - smartphones
25 - tablets
26 - laptops
27 - monitors
28 - televisions
29 - plant research
30 - energy research
31 - mining research
32 - electronics research
33 - breeding research
34 - chemistry research
35 - software
36 - NONE / BLANK PAGE
37 - NONE / BLANK PAGE
38 - NONE / BLANK PAGE
39 - NONE / BLANK PAGE
40 - cotton
41 - fabric
42 - iron ore
43 - steel
44 - sand
45 - glass
46 - leather
47 - on-board computer
48 - electric motor
49 - luxury car interior
50 - basic interior
51 - car body
52 - combustion engine
53 - economy e-car
54 - luxury e-car
55 - economy car
56 - luxury car
57 - truck
58 - automotive research
59 - fashion research
60 - underwear
61 - gloves
62 - dress
63 - stiletto heel
64 - handbags
65 - sneakers
66 - seeds
67 - xmas crackers
68 - gold ore
69 - golden bars
70 - luxury watch
71 - necklace
72 - sugarcane
73 - ethanol
74 - methane
75 - carbon fibers
76 - carbon composite
77 - fuselage
78 - wing
79 - high grade e-comps
80 - flight computer
81 - cockpit
82 - attitude control
83 - rocket fuel
84 - propellant tank
85 - solid fuel booster
86 - rocket engine
87 - heat shield
88 - ion drive
89 - jet engine
90 - sub-orbital 2nd stage
91 - sub-orbital rocket
92 - orbital booster
93 - starship
94 - bfr
95 - jumbo jet
96 - luxury jet
97 - single engine plane
98 - quadcopter
99 - satellite
100 - aerospace research
101 - reinforced concrete
102 - bricks
103 - cement
104 - clay
105 - limestone
106 - wood
107 - beams
108 - planks
109 - windows
110 - tools
111 - construction units
112 - bulldozer
113 - materials research
114 - robots
115 - cows
116 - pigs
117 - milk
118 - coffee beans
119 - coffee powder
120 - vegetables
121 - bread
122 - cheese
123 - apple pie
124 - orange juice
125 - apple cider
126 - ginger beer
127 - frozen pizza
128 - pasta
129 - hamburger
130 - lasagna
131 - meat balls
132 - cocktails
133 - flour
134 - butter
135 - sugar
136 - cocoa
137 - dough
138 - sauce
139 - fodder
140 - chocolate
141 - vegetable oil
142 - salad
143 - samosa
144 - xmas ornament
145 - recipes
146 - pumpkin
147 - jack o'lantern
148 - witch costume
149 - pumpkin soup
150 - tree
151 - easter bunny
152 - ramadan sweets
153 - chocolate icecream
154 - apple icecream
155 - cream egg'''


In [15]:
file_path = "../../data/source/Book1.xlsx"

df_products = pd.read_excel(
    file_path,
    sheet_name="products"
)

print(df_products.head())

   product_id     name     category
0           1    Seeds  Agriculture
1           2   Apples  Agriculture
2           3  Oranges  Agriculture
3           4   Grapes  Agriculture
4           5    Grain  Agriculture


In [16]:
market_ids = market_ids.split('\n')
market_ids[2]

'3 - apples'

In [17]:
df_market = pd.DataFrame()

for market_id in market_ids:
    id, name = market_id.split('-', 1)
    id = id.strip()
    name = name.strip().lower()
    
    new_row = pd.DataFrame([{'market_id':id, 'name':name}])
    df_market = pd.concat([df_market, new_row], ignore_index=True)

In [18]:
df_market.head()

,market_id,name
0,1,power
1,2,water
2,3,apples
3,4,oranges
4,5,grapes


In [19]:
df_products['name'] = df_products['name'].str.strip().str.lower()

In [20]:
df_mapped = pd.merge(
    df_market,
    df_products,
    on='name',
    how='outer',
    suffixes=('_market', '_product'),
    indicator=True
)

In [21]:
df_market_unmatched = df_mapped[
    df_mapped['_merge'] == 'left_only'
]

df_products_unmatched = df_mapped[
    df_mapped['_merge'] == 'right_only'
]

df_matched = df_mapped[
    df_mapped['_merge'] == 'both'
]

In [22]:
print("=== MAPPING SUMMARY ===")
print(df_mapped['_merge'].value_counts())

print("\n=== MARKET WITHOUT MATCH ===")
print(df_market_unmatched[['market_id', 'name']].to_string(index=False))

print("\n=== PRODUCTS WITHOUT MARKET ID ===")
print(df_products_unmatched[['product_id', 'name', 'category']].to_string(index=False))

=== MAPPING SUMMARY ===
_merge
both          151
left_only       4
right_only      0
Name: count, dtype: int64

=== MARKET WITHOUT MATCH ===
market_id              name
       36 none / blank page
       37 none / blank page
       38 none / blank page
       39 none / blank page

=== PRODUCTS WITHOUT MARKET ID ===
Empty DataFrame
Columns: [product_id, name, category]
Index: []


In [23]:
df_mapped = pd.merge(
    df_market,
    df_products,
    on='name',
    how='inner',
    suffixes=('_market', '_product'),
    indicator=True
)

In [24]:
with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    df_mapped.to_excel(
        writer,
        sheet_name="product_mapping",
        index=False
    )